# BLIP-2 Fine-Tune

In [ ]:
# Cell 1: Imports and Basic Setup
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from PIL import Image
from tqdm import tqdm
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import Blip2Processor, Blip2ForConditionalGeneration

In [ ]:
# Cell 2: Data Loading
from google.colab import drive
drive.mount('/content/drive')
!unzip -q "/content/drive/MyDrive/caption_dataset/caption_dataset.zip" -d "/content/dataset/"

DATASET_PATH = "/content/dataset/"
TRAIN_CSV = os.path.join(DATASET_PATH, "train.csv")
TEST_CSV = os.path.join(DATASET_PATH, "test.csv")
TRAIN_IMAGE_DIR = os.path.join(DATASET_PATH, "train", "train")
TEST_IMAGE_DIR = os.path.join(DATASET_PATH, "test", "test")

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

In [ ]:
# Cell 3: Fine-Tune BLIP (public BLIP-1 base model)

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.cuda.amp import GradScaler, autocast
import torch.multiprocessing as mp
from transformers import BlipProcessor, BlipForConditionalGeneration

mp.set_start_method('spawn', force=True)

# Public BLIP-1 base model
processor_blip = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
blip_model.to(device)


train_df = train_df.sample(n=2000, random_state=42)
train_df['caption_proc'] = train_df['caption'].apply(lambda x: "<start> " + x.strip() + " <end>")

class CustomDataset(Dataset):
    def __init__(self, processor, image_ids, captions, image_dir):
        self.processor = processor
        self.image_paths = [os.path.join(image_dir, f"{img_id}.jpg") for img_id in image_ids]
        self.captions = captions
    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, idx):
        try:
            image = Image.open(self.image_paths[idx]).convert("RGB").resize((32, 32))
        except:
            image = Image.new("RGB", (32, 32), color='white')
        encoding = self.processor(images=image, text=self.captions[idx], return_tensors="pt", padding="max_length", truncation=True, max_length=32)
        encoding = {k: v.squeeze(0).to(device) for k, v in encoding.items()}
        return encoding

dataset = CustomDataset(processor_blip, train_df['image_id'].tolist(), train_df['caption_proc'].tolist(), TRAIN_IMAGE_DIR)
dataloader = DataLoader(dataset, batch_size=40, shuffle=True, num_workers=0)

optimizer = AdamW(blip_model.parameters(), lr=5e-5)
scaler = GradScaler()
num_epochs = 3
blip_model.train()

for epoch in range(num_epochs):
    running_loss = 0.0
    for batch in tqdm(dataloader, desc=f"Epoch {epoch+1}"):
        optimizer.zero_grad()
        with autocast():
            outputs = blip_model(
                input_ids=batch["input_ids"],
                pixel_values=batch["pixel_values"],
                attention_mask=batch["attention_mask"],
                labels=batch["input_ids"]
            )
            loss = outputs.loss
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
    print(f"Epoch {epoch+1} Completed. Average Loss: {running_loss/len(dataloader):.4f}")

blip_model.save_pretrained("./blip-finetuned")
processor_blip.save_pretrained("./blip-finetuned")
print("Fine-tuned BLIP-1 base (public) and processor saved.")


In [ ]:
# Cell 4: Prediction

import pandas as pd
from PIL import Image
import os


DATASET_PATH = "/content/dataset"
TEST_CSV = os.path.join(DATASET_PATH, "test.csv")
TEST_IMAGE_DIR = os.path.join(DATASET_PATH, "test", "test")

# Test CSV
test_df = pd.read_csv(TEST_CSV)

# BLIP-1 model evaluation
blip_model.eval()
preds = []
for img_id in test_df['image_id'].tolist():
    img_path = os.path.join(TEST_IMAGE_DIR, f"{img_id}.jpg")
    try:
        image = Image.open(img_path).convert("RGB").resize((32, 32))
        inputs = processor_blip(images=image, return_tensors="pt").to(device)
        outputs = blip_model.generate(**inputs)
        caption = processor_blip.batch_decode(outputs, skip_special_tokens=True)[0]
    except:
        caption = "Image not found"
    preds.append({'image_id': img_id, 'caption': caption})

submission_df = pd.DataFrame(preds)
submission_df.to_csv("submission_blip_finetuned.csv", index=False)
print("Prediction CSV (submission_blip_finetuned.csv) saved.")


In [ ]:
"""
This metric implements the mean Fréchet GTE Distance (FGD) score for text embeddings using the GTE-small model.
The metric measures the similarity between ground truth and predicted text captions by comparing their embedding distributions.

The score is calculated by:
1. Converting text to embeddings using GTE-small
2. Computing mean and covariance statistics of the embeddings
3. Calculating the FGD score between the distributions
"""

import pandas as pd
import pandas.api.types
import numpy as np
from numpy import cov, trace, iscomplexobj
from scipy.linalg import sqrtm
from typing import List
from sentence_transformers import SentenceTransformer

def calculate_fgd(solution_embed: np.ndarray, submission_embed: np.ndarray) -> float:
    '''
    solution_embed: Embedding of the ground truth from GTE-small.
    submission_embed: Embedding of the predicted caption from GTE-small.
    '''
    fgd_list = []
    for _idx, (sol_emb_sample, sub_emb_sample) in enumerate(zip(solution_embed, submission_embed)):
        sol_emb_sample_rshaped, sub_emb_sample_rshaped = sol_emb_sample.reshape((1,384)), sub_emb_sample.reshape((1,384))
        e1 = np.concatenate([sol_emb_sample_rshaped, sol_emb_sample_rshaped])
        e2 = np.concatenate([sub_emb_sample_rshaped, sub_emb_sample_rshaped])
        """Calculate Fréchet GTE Distance between two embedding distributions"""
        # Calculate mean and covariance statistics
        mu1, sigma1 = e1.mean(axis=0), cov(e1, rowvar=False)
        mu2, sigma2 = e2.mean(axis=0), cov(e2, rowvar=False)

        # Calculate sum squared difference between means
        ssdiff = np.sum((mu1 - mu2)**2.0)

        # Calculate sqrt of product between cov
        covmean = sqrtm(sigma1.dot(sigma2))

        # Check and correct imaginary numbers from sqrt
        if iscomplexobj(covmean):
            covmean = covmean.real

        # Calculate score
        fgd = ssdiff + trace(sigma1 + sigma2 - 2.0 * covmean)
        fgd_list.append(fgd)
        if _idx % 100 == 0:
            print(f"Processed {_idx} samples", end="\r")
    return float(np.mean(fgd_list))

In [ ]:
from sentence_transformers import SentenceTransformer
import pandas as pd

# GTE-small model
gte_model = SentenceTransformer("thenlper/gte-small")

# Ground truth captions
gt_df = pd.read_csv(TRAIN_CSV)
gt_captions = gt_df["caption"].values
gt_embed = gte_model.encode(list(gt_captions), batch_size=64, show_progress_bar=True)

# Fine-tuned BLIP output
submission_df = pd.read_csv("submission_blip_finetuned.csv")
N = min(len(gt_df), len(submission_df))
pred_captions = submission_df["caption"].values[:N]
pred_embed = gte_model.encode(list(pred_captions), batch_size=64, show_progress_bar=True)

# FGD hesapla
fgd_score = calculate_fgd(gt_embed[:N], pred_embed)
print(f"\nFGD Score for BLIP-1 (fine-tuned): {fgd_score:.5f}")


In [ ]:
import matplotlib.pyplot as plt
import os
from PIL import Image
import pandas as pd

TEST_IMAGE_DIR = "/content/dataset/test/test"

# Fine-tuned BLIP output
submission_df = pd.read_csv("submission_blip_finetuned.csv")

print("\nRandom 50 Sample Outputs with Images (BLIP Fine-Tuned):\n")
sample_df = submission_df.sample(50, random_state=42)

rows, cols = 10, 5
fig, axes = plt.subplots(rows, cols, figsize=(20, 40))
axes = axes.flatten()

for ax, (_, row) in zip(axes, sample_df.iterrows()):
    img_id = str(row['image_id'])
    if not img_id.endswith(".jpg"):
        img_id += ".jpg"
    img_path = os.path.join(TEST_IMAGE_DIR, img_id)
    try:
        image = Image.open(img_path).convert("RGB")
        ax.imshow(image)
        ax.axis("off")
        ax.set_title(f"{row['caption']}", fontsize=6)
    except FileNotFoundError:
        ax.axis("off")
        ax.set_title("Image not found", fontsize=6)

plt.tight_layout()
plt.show()